In [2]:
import sys
sys.executable

'/usr/local/opt/python@3.11/bin/python3.11'

In [3]:
import ast
import pandas as pd
import numpy as np
from pygit2 import Object, Repository, GIT_SORT_TIME
from pygit2 import init_repository, Patch
from colorama import Fore
from tqdm import tqdm
import swifter
from pandarallel import pandarallel
from bs4 import BeautifulSoup, SoupStrainer
import requests
from urllib.request import urlopen
import re
import time
import random
import subprocess
import os

In [4]:
pandarallel.initialize(progress_bar=True)

INFO: Pandarallel will run on 8 workers.
INFO: Pandarallel will use standard multiprocessing data transfer (pipe) to transfer data between the main process and workers.


In [5]:
library = 'lablup/backend.ai'

In [6]:
# import all push data
df_pr = pd.DataFrame()
commit_urls = []
for val in np.arange(0, 100, 1):
    if val < 10:
        val = "0" + str(val)
    try:
        df_part = pd.read_csv(f'data/github_clean/prEventCommits0000000000{val}.csv', index_col = 0)
        df_part['partition'] = val
        df_pr = pd.concat([df_pr, df_part])
    except:
        print(f'data/github_clean/prEventCommits0000000000{val}.csv not found')

data/github_clean/prEventCommits000000000019.csv not found
data/github_clean/prEventCommits000000000025.csv not found
data/github_clean/prEventCommits000000000026.csv not found
data/github_clean/prEventCommits000000000027.csv not found
data/github_clean/prEventCommits000000000028.csv not found
data/github_clean/prEventCommits000000000029.csv not found


In [7]:
df_library = df_pr[df_pr['repo_name'] == library]

In [58]:
def getHead(commit_list, pull_number):
    try:
        commit = ast.literal_eval(commit_list)[0]
        pull_fetch = subprocess.Popen(["git","fetch", "origin", f"pull/{pull_number}/head"], cwd = "repos/backend.ai").wait()
        result = subprocess.run(["git","show", f"{commit}^"], cwd = "repos/backend.ai", capture_output = True, text = True).stdout[7:47]
        return result
    except Exception as e:
        return str(e)

In [51]:
repo = Repository('repos/backend.ai')

In [120]:
def createCommitGroup(commit_list, parent_commit):
    try:
        commit_list = ast.literal_eval(commit_list)
        if len(commit_list) == 0:
            return [[]]
        elif len(commit_list) == 1:
            return [[parent_commit, commit_list[0]]]
        else:
            avail_commits = len(commit_list)
            lst_result = [[parent_commit, commit_list[0]]]
            for i in range(avail_commits-1):
                lst_result.append([commit_list[i], commit_list[i+1]])
            return lst_result
    except:
        return [[]]

In [102]:
df_library[['pr_number', 'repo_id', 'repo_name', 'actor_id', 'actor_login', 
                                   'org_id', 'org_login', 'pr_state', 'commit_list']].explode('commit_groups')


709.0       5246c2efe6e74f3f5620cd9fa1cde850b68b5af2
4024.0      b8180606b17dd16724a7e5e97b863dd7e8a76243
6020.0      d0379be63995494c53927b7a117710e50c910ed0
9763.0      10ff4c680f1beb0a02b42b2d582977e9f11f28e1
20133.0     27eee4f8d2322d0f1ff0a2711949be0f1dcf0e52
                              ...                   
95663.0     94aacb5ece8b04a1a6f803e580a102772d387a91
100277.0    f6d21825414736fd95f8b6c47e3e97d4e90d0b9e
103807.0    d27789a2eb5aa46c0c4002ed05c392affc240bc8
106595.0    501cce10676f26057bbdb6afa39fd9fbc89418dc
112112.0    0d41342c306f90aaecd0636c62bd8ac0bdc47efa
Length: 1608, dtype: object

In [121]:
df_library['commit_groups'] = df_library.parallel_apply(lambda x: createCommitGroup(x['commit_list'], x['parent_commit']), axis = 1)


/var/folders/5s/dvrxt95949x1pm_sjxm85lj00000gn/T/ipykernel_64457/2988718724.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_library['commit_groups'] = df_library.parallel_apply(lambda x: createCommitGroup(x['commit_list'], x['parent_commit']), axis = 1)


In [154]:
#!/usr/bin/env python
# coding: utf-8

# In[322]:
import sys
sys.executable


# In[381]:
import ast
import pandas as pd
import numpy as np
from pygit2 import Object, Repository, GIT_SORT_TIME
from pygit2 import init_repository, Patch
from colorama import Fore
from tqdm import tqdm
import swifter
from pandarallel import pandarallel
import subprocess
import warnings
from joblib import Parallel, delayed
import os
import multiprocessing
import time
import random


# In[388]:
def createCommitGroup(commit_list, parent_commit):
    try:
        commit_list = ast.literal_eval(commit_list)
        if len(commit_list) == 0:
            return [[]]
        elif len(commit_list) == 1:
            return [[parent_commit, commit_list[0]]]
        else:
            avail_commits = len(commit_list)
            lst_result = [[parent_commit, commit_list[0]]]
            for i in range(avail_commits-1):
                lst_result.append([commit_list[i], commit_list[i+1]])
            return lst_result
    except:
        return [[]]



def getHead(commit_list, pull_number, repo_loc):
    try:
        commit = ast.literal_eval(commit_list)[0]
        pull_fetch = subprocess.Popen(["git","fetch", "origin", f"pull/{pull_number}/head"], cwd = f"{repo_loc}").wait()
        result = subprocess.run(["git","show", f"{commit}^"], cwd = f"{repo_loc}", capture_output = True, text = True).stdout[7:47]
        return result
    except Exception as e:
        return []


# In[390]:
def returnCommitStats(x):
    if len(x) < 2:
        return []
    if x[0] == [] or x[1] == []:
        return []
    commit_parent_sha = x[0]
    commit_head_sha = x[1]
    commit_parent = repo.get(commit_parent_sha)
    commit_head = repo.get(commit_head_sha)
    if type(commit_parent) != type(None) and type(commit_head) != type(None):
        diff = repo.diff(commit_parent, commit_head, context_lines=0, interhunk_lines=0)
        commit_sha = commit_head_sha
        commit_author_name = commit_head.author.name
        commit_author_email = commit_head.author.email
        committer_author_name = commit_head.committer.name
        committer_author_email = commit_head.committer.email
        commit_message = commit_head.message
        commit_additions = diff.stats.insertions
        commit_deletions = diff.stats.deletions
        commit_changes_total = commit_additions + commit_deletions
        commit_files_changed_count = diff.stats.files_changed
        commit_time = commit_head.commit_time
        commit_file_changes = []
        
        for obj in diff:
            if type(obj) == Patch:
                additions = 0
                deletions = 0
                for hunk in obj.hunks:
                  for line in hunk.lines:
                    # The new_lineno represents the new location of the line after the patch. If it's -1, the line has been deleted.
                    if line.new_lineno == -1: 
                        deletions += 1
                    # Similarly, if a line did not previously have a place in the file, it's been added fresh. 
                    if line.old_lineno == -1: 
                        additions += 1
                commit_file_changes.append({'file':obj.delta.new_file.path,
                                            'additions': additions,
                                            'deletions': deletions,
                                            'total': additions + deletions})
        return [commit_sha, commit_author_name, commit_author_email, committer_author_name, committer_author_email,
                commit_message, commit_additions, commit_deletions, commit_changes_total, commit_files_changed_count,
                commit_file_changes]
    return []
        
def cleanCommitData(library, repo_loc):
    # In[386]:
    df_library = df_pr[df_pr['repo_name'] == library]

    # In[387]:
    global repo
    repo = Repository(repo_loc)

    df_library['parent_commit'] = df_library.parallel_apply(lambda x: getHead(x['commit_list'], x['pr_number'], repo_loc), axis = 1)
    print(f"finished getting parent commits for {library}")
    df_library['commit_groups'] = \
        df_library.parallel_apply(lambda x: createCommitGroup(x['commit_list'], x['parent_commit']), axis = 1)

    df_commit_groups = df_library[['pr_number', 'repo_id', 'repo_name', 'actor_id', 'actor_login', 
                                   'org_id', 'org_login', 'pr_state', 'commit_groups']].explode('commit_groups')
    df_commit_groups = df_commit_groups[df_commit_groups['commit_groups'].apply(lambda x: len(x)>0)]
    
    commit_data = df_commit_groups['commit_groups'].parallel_apply(lambda x: returnCommitStats(x))

    # In[ ]:
    df_commit = pd.DataFrame(commit_data.tolist(),
                            columns = ['commit sha', 'commit author name', 'commit author email', 'committer name',
                                       'commmitter email', 'commit message', 'commit additions', 'commit deletions',
                                       'commit changes total', 'commit files changed count', 'commit file changes'])
    
    
    # In[ ]:
    df_commit_final = pd.concat([df_commit_groups.reset_index(drop = True), df_commit], axis = 1)
    return df_commit_final


def getCommitData(library):
    # download repo
    lib_p2 = library.split("/")[1]
    lib_ren = library.replace("/","___")
    if f'commits_pr_{lib_ren}.parquet' not in os.listdir('data/github_commits/parquet') and f'{lib_ren}' not in os.listdir('repos'):
        try:
            print(f"Starting {library}")
            start = time.time()
            if lib_ren not in os.listdir("repos"):
                subprocess.Popen(["git", "clone", f"https://github.com/{library}.git", f"{lib_ren}"], cwd = "repos").communicate()
            print(f"Finished cloning {library}")
            df_lib = cleanCommitData(library, f"repos/{lib_ren}")
            df_lib.to_parquet(f'data/github_commits/parquet/commits_pr_{lib_ren}.parquet')
            end = time.time()
            subprocess.Popen(["rm", "-rf", f"{lib_ren}"], cwd = "repos").communicate()
            print(f"{library} completed in {start - end}")
            return "success"
        except Exception as e:
            return f"failure, {str(e)}"
    return 'success'


In [155]:
library = 'python-openapi/openapi-core'
lib_p2 = library.split("/")[1]
lib_ren = library.replace("/","___")
repo_loc = f'repos/{lib_ren}'

In [156]:
df_library = df_pr[df_pr['repo_name'] == library]

# In[387]:
global repo
repo = Repository(repo_loc)

df_library['parent_commit'] = df_library.parallel_apply(lambda x: getHead(x['commit_list'], x['pr_number'], repo_loc), axis = 1)
print(f"finished getting parent commits for {library}")
df_library['commit_groups'] = \
    df_library.parallel_apply(lambda x: createCommitGroup(x['commit_list'], x['parent_commit']), axis = 1)

df_commit_groups = df_library[['pr_number', 'repo_id', 'repo_name', 'actor_id', 'actor_login', 
                               'org_id', 'org_login', 'pr_state', 'commit_groups']].explode('commit_groups')
df_commit_groups = df_commit_groups[df_commit_groups['commit_groups'].apply(lambda x: len(x)>0)]

From https://github.com/python-openapi/openapi-core
 * branch            refs/pull/627/head -> FETCH_HEAD
From https://github.com/python-openapi/openapi-core
 * branch            refs/pull/587/head -> FETCH_HEAD
From https://github.com/python-openapi/openapi-core
 * branch            refs/pull/561/head -> FETCH_HEAD
From https://github.com/python-openapi/openapi-core
 * branch            refs/pull/584/head -> FETCH_HEAD
From https://github.com/python-openapi/openapi-core
 * branch            refs/pull/633/head -> FETCH_HEAD
From https://github.com/python-openapi/openapi-core
 * branch            refs/pull/590/head -> FETCH_HEAD
From https://github.com/python-openapi/openapi-core
 * branch            refs/pull/606/head -> FETCH_HEAD
From https://github.com/python-openapi/openapi-core
 * branch            refs/pull/541/head -> FETCH_HEAD
Process ForkPoolWorker-239:
Process ForkPoolWorker-237:
From https://github.com/python-openapi/openapi-core
 * branch            refs/pull/622/head -> F

KeyboardInterrupt: 

Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
  File "/usr/local/Cellar/python@3.11/3.11.5/Frameworks/Python.framework/Versions/3.11/lib/python3.11/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/usr/local/Cellar/python@3.11/3.11.5/Frameworks/Python.framework/Versions/3.11/lib/python3.11/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/Cellar/python@3.11/3.11.5/Frameworks/Python.framework/Versions/3.11/lib/python3.11/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/usr/local/Cellar/python@3.11/3.11.5/Frameworks/Python.framework/Versions/3.11/lib/python3.11/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
From https://github.com/python-openapi/openapi-core
 * branch            refs/pull/623/head -> FETCH_HEAD
From https://github.com/python-openapi/openapi-core
From https://github.com/python-op

In [ ]:
repos = df_pr['repo_name'].unique().tolist()
results = []
#repos = ['lablup/backend.ai']

for r in repos:
    result = getCommitData(r)
    print(r, result)
    results.append(result)
    
print("Done!")

Starting python-openapi/openapi-core


Cloning into 'python-openapi___openapi-core'...


Finished cloning python-openapi/openapi-core


From https://github.com/python-openapi/openapi-core
 * branch            refs/pull/587/head -> FETCH_HEAD
From https://github.com/python-openapi/openapi-core
 * branch            refs/pull/627/head -> FETCH_HEAD
From https://github.com/python-openapi/openapi-core
 * branch            refs/pull/590/head -> FETCH_HEAD
From https://github.com/python-openapi/openapi-core
 * branch            refs/pull/606/head -> FETCH_HEAD
From https://github.com/python-openapi/openapi-core
 * branch            refs/pull/633/head -> FETCH_HEAD
From https://github.com/python-openapi/openapi-core
 * branch            refs/pull/541/head -> FETCH_HEAD
From https://github.com/python-openapi/openapi-core
 * branch            refs/pull/584/head -> FETCH_HEAD
From https://github.com/python-openapi/openapi-core
 * branch            refs/pull/561/head -> FETCH_HEAD
From https://github.com/python-openapi/openapi-core
 * branch            refs/pull/584/head -> FETCH_HEAD
From https://github.com/python-openapi/openapi

In [98]:
t = df_pr.drop('partition',axis=1).drop_duplicates()
df_pr = df_pr.loc[t.index]

In [66]:
head.loc[103537.0]

'ed87e31bfd9811e1b29b67c2547918dbe4530a69'

In [59]:
%%time
head = df_library.parallel_apply(lambda x: getHead(x['commit_list'], x['pr_number']), axis = 1)

From https://github.com/lablup/backend.ai
 * branch              refs/pull/1212/head -> FETCH_HEAD
Auto packing the repository in background for optimum performance.
See "git help gc" for manual housekeeping.
and remove .git/gc.log.
Automatic cleanup will not be performed until the file is removed.


From https://github.com/lablup/backend.ai
 * branch              refs/pull/1513/head -> FETCH_HEAD
From https://github.com/lablup/backend.ai
 * branch              refs/pull/697/head -> FETCH_HEAD
Auto packing the repository in background for optimum performance.
See "git help gc" for manual housekeeping.
and remove .git/gc.log.
Automatic cleanup will not be performed until the file is removed.


Auto packing the repository in background for optimum performance.
See "git help gc" for manual housekeeping.
and remove .git/gc.log.
Automatic cleanup will not be performed until the file is removed.


From https://github.com/lablup/backend.ai
 * branch              refs/pull/1231/head -> FETCH_H

CPU times: user 1.22 s, sys: 863 ms, total: 2.08 s
Wall time: 3min 5s


In [387]:
repo = Repository('../repos/backend.ai')
sum = 0
for commit in commit_urls:
    if type(commit) != float:
        sha = commit.split("/")[-1]
        commit_a = repo.get(sha)
        if type(commit_a) != type(None):
            sum += 1
print(f"{sum} out of {len(commit_urls)} commit SHAs can be found ({100*round(sum/len(commit_urls), 3)}%)")

21523 out of 25698 commit SHAs can be found (83.8%)


In [388]:
def createCommitGroup(push_before, push_head, commit_urls, push_size):
    if push_size == 0:
        return [[]]
    elif push_size == 1:
        return [[push_before, push_head]]
    else:
        avail_commits = len(commit_urls)
        lst_result = [[push_before, commit_urls[0].split("/")[-1]]]
        for i in range(avail_commits-1):
            lst_result.append([commit_urls[i].split("/")[-1], commit_urls[i+1].split("/")[-1]])
        return lst_result

In [389]:
df_library['commit_groups'] = \
    df_library.apply(lambda x: createCommitGroup(x['push_before'], x['push_head'], x['commit_urls'], x['push_size']), axis = 1)
df_commit_groups = df_library[['push_id', 'repo_id', 'repo_name', 'actor_id', 'actor_login', 
                               'org_id', 'org_login', 'push_size', 'push_before', 'push_head',
                               'commit_groups']].explode('commit_groups')

/var/folders/5s/dvrxt95949x1pm_sjxm85lj00000gn/T/ipykernel_39288/1534266405.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_library['commit_groups'] = \


In [390]:
def returnCommitStats(x):
    if len(x) < 2:
        return []
    commit_parent_sha = x[0]
    commit_head_sha = x[1]
    commit_parent = repo.get(commit_parent_sha)
    commit_head = repo.get(commit_head_sha)
    if type(commit_parent) != type(None) and type(commit_head) != type(None):
        diff = repo.diff(commit_parent, commit_head, context_lines=0, interhunk_lines=0)
        commit_sha = commit_head_sha
        commit_author_name = commit_head.author.name
        commit_author_email = commit_head.author.email
        committer_author_name = commit_head.committer.name
        committer_author_email = commit_head.committer.email
        commit_message = commit_head.message
        commit_additions = diff.stats.insertions
        commit_deletions = diff.stats.deletions
        commit_changes_total = commit_additions + commit_deletions
        commit_files_changed_count = diff.stats.files_changed
        commit_time = commit_head.commit_time
        commit_file_changes = []
        
        for obj in diff:
            if type(obj) == Patch:
                additions = 0
                deletions = 0
                for hunk in obj.hunks:
                  for line in hunk.lines:
                    # The new_lineno represents the new location of the line after the patch. If it's -1, the line has been deleted.
                    if line.new_lineno == -1: 
                        deletions += 1
                    # Similarly, if a line did not previously have a place in the file, it's been added fresh. 
                    if line.old_lineno == -1: 
                        additions += 1
                commit_file_changes.append({'file':obj.delta.new_file.path,
                                            'additions': additions,
                                            'deletions': deletions,
                                            'total': additions + deletions})
        return [commit_sha, commit_author_name, commit_author_email, committer_author_name, committer_author_email,
                commit_message, commit_additions, commit_deletions, commit_changes_total, commit_files_changed_count,
                commit_file_changes]
    return []

In [392]:
%%time
commit_data = df_commit_groups['commit_groups'].parallel_apply(returnCommitStats)

CPU times: user 1.54 s, sys: 1.02 s, total: 2.56 s
Wall time: 12min 53s


In [393]:
df_commit = pd.DataFrame(commit_data.tolist(),
                        columns = ['commit sha', 'commit author name', 'commit author email', 'committer name',
                                   'commmitter email', 'commit message', 'commit additions', 'commit deletions',
                                   'commit changes total', 'commit files changed count', 'commit file changes'])

In [394]:
df_commit_final = pd.concat([df_commit_groups.reset_index(drop = True), df_commit], axis = 1)
df_commit_final

,push_id,repo_id,repo_name,actor_id,actor_login,org_id,org_login,push_size,push_before,push_head,...,commit author name,commit author email,committer name,commmitter email,commit message,commit additions,commit deletions,commit changes total,commit files changed count,commit file changes
0,11550771535,70124554,lablup/backend.ai,2904494,kyujin-cho,11258248.0,lablup,84,f7288e25e5ba7e30ed58946942ea52413912d0bd,e27a7900fe25a6e392a3ff35da593d071422d65b,...,None,None,None,None,None,NaN,NaN,NaN,NaN,None
1,11550771535,70124554,lablup/backend.ai,2904494,kyujin-cho,11258248.0,lablup,84,f7288e25e5ba7e30ed58946942ea52413912d0bd,e27a7900fe25a6e392a3ff35da593d071422d65b,...,Joongi Kim,joongi@lablup.com,GitHub,noreply@github.com,setup: Remove dependency to psycopg (#702)\n\n...,204.0,296.0,500.0,21.0,"[{'file': 'changes/702.breaking.md', 'addition..."
2,11550771535,70124554,lablup/backend.ai,2904494,kyujin-cho,11258248.0,lablup,84,f7288e25e5ba7e30ed58946942ea52413912d0bd,e27a7900fe25a6e392a3ff35da593d071422d65b,...,Sanghun Lee,oper6909@gmail.com,GitHub,noreply@github.com,feat: Do not return AgentSummary if `hide-agen...,10.0,2.0,12.0,2.0,"[{'file': 'changes/699.feature.md', 'additions..."
3,11550771535,70124554,lablup/backend.ai,2904494,kyujin-cho,11258248.0,lablup,84,f7288e25e5ba7e30ed58946942ea52413912d0bd,e27a7900fe25a6e392a3ff35da593d071422d65b,...,Jonghyun Park,jpark@lablup.com,GitHub,noreply@github.com,feat: add `hide_agents` webserver config (#704...,6.0,1.0,7.0,4.0,"[{'file': 'changes/704.feature', 'additions': ..."
4,11550771535,70124554,lablup/backend.ai,2904494,kyujin-cho,11258248.0,lablup,84,f7288e25e5ba7e30ed58946942ea52413912d0bd,e27a7900fe25a6e392a3ff35da593d071422d65b,...,DaeHyun Sung,dhsung@lablup.com,GitHub,noreply@github.com,docs: update python default version (#724)\n\n...,1448.0,996.0,2444.0,11.0,"[{'file': 'docs/README.md', 'additions': 1, 'd..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25693,11646000079,70124554,lablup/backend.ai,555156,achimnol,11258248.0,lablup,286,6f1c5a2c203da153978b9b334709b33d6ba0383a,a2e85fd58c9d7fe10404bc26f6966f20c9097040,...,Jeongseok Kang,piono623@naver.com,GitHub,noreply@github.com,fix: Update integration test to work with the ...,96.0,55.0,151.0,8.0,"[{'file': 'changes/442.fix.md', 'additions': 1..."
25694,13838497595,70124554,lablup/backend.ai,2904494,kyujin-cho,11258248.0,lablup,1,437e4625f172057e645465c7084492cfeb0143e8,8e26fd42b6ec476b084f237b3b7a4a5c2f8e0c8f,...,Kyujin Cho,kyujin.cho@lablup.com,GitHub,noreply@github.com,feat: update POST `/service` API to also accep...,85.0,38.0,123.0,3.0,[{'file': 'src/ai/backend/client/cli/service.p...
25695,10610526310,70124554,lablup/backend.ai,555156,achimnol,11258248.0,lablup,1,37c8df3174c768b96f542e6511f33ffad219079e,b1c7115142f78f43743f85f417f1f7c852116dc5,...,None,None,None,None,None,NaN,NaN,NaN,NaN,None
25696,10753912953,70124554,lablup/backend.ai,555156,achimnol,11258248.0,lablup,1,6eb9396d72c19b8388a1dec5f9749192f59c1f97,cc5fc1d3b750ad1cae16d18d01544efcc2453caf,...,None,None,None,None,None,NaN,NaN,NaN,NaN,None


In [418]:
df_repos

,github repo,count
0,dagster-io/dagster,43538
1,Azure/azure-sdk-for-python,30160
2,DataDog/dd-trace-py,26048
3,pulumi/pulumi,21919
4,ray-project/ray,20780
...,...,...
5846,hellock/cvbase,1
5847,heroku/heroku.py,1
5848,DenisCarriere/geocoder,1
5849,nickoala/telepot,1


In [417]:
df_repos = pd.DataFrame(df_push[['repo_name']].value_counts()).reset_index().rename(
    {'repo_name': 'github repo'}, axis = 1)
df_repos.to_csv('data/inputs/github_repo_grab_commits.csv')

In [430]:
import subprocess
lib = 'dagster-io/dagster.git'
lib_p2 = lib.split("/")[1]
lib_ren = lib.replace("/","___")
subprocess.Popen(["git", "clone", f"https://github.com/{lib}", f"{lib_ren}"], cwd = "../repos").wait()

Cloning into 'dagster-io___dagster.git'...
Updating files: 100% (7050/7050), done.


0